In [ ]:
"""BigAlpha 2026 factor submission.

The competition runner imports ``main(datasources, start_date, end_date)``.
This file is also mirrored into the deliverable notebook.

AI-track disclosure:
    The factor expression below was generated and screened by an AI FactorMiner
    workflow.  The deterministic compiler converts that expression to DAI SQL;
    no model call, external network access, or future data is used at evaluation
    time.
"""

import ast
import re


# One BigAlpha submission must contain exactly one factor.  To submit a different
# entry from factors.json, replace these three constants and keep the rest intact.
FACTOR_ID = "ma5_ma20_deviation"
FACTOR_EXPRESSION = "div(sub(ts_mean(close, 5), ts_mean(close, 20)), ts_mean(close, 20))"
TRAIN_IC = -0.040611

# The largest window in factors.json is 30 trading days.  The official evaluator
# exposes roughly one quarter of earlier data; requesting 100 calendar days makes
# that history available to the time-series operators at the evaluation boundary.
BUFFER_DAYS = 100


_ALLOWED_FIELDS = {
    "open",
    "high",
    "low",
    "close",
    "volume",
    "daily_return",
}


def _expression_to_sql(expression):
    """Compile a validated FactorMiner expression into DAI SQL."""

    def compile_node(node):
        if isinstance(node, ast.Constant):
            if isinstance(node.value, bool):
                return "TRUE" if node.value else "FALSE"
            if isinstance(node.value, (int, float)):
                return repr(node.value)
            raise ValueError("Only numeric and Boolean constants are allowed")

        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            return f"(-({compile_node(node.operand)}))"

        if isinstance(node, ast.Name):
            if node.id not in _ALLOWED_FIELDS:
                raise ValueError(f"Unsupported field: {node.id}")
            return node.id

        if not isinstance(node, ast.Call) or not isinstance(node.func, ast.Name):
            raise ValueError(f"Unsupported expression node: {ast.dump(node)}")
        if node.keywords:
            raise ValueError("Keyword arguments are not allowed")

        name = node.func.id
        args = [compile_node(arg) for arg in node.args]

        def require(n):
            if len(args) != n:
                raise ValueError(f"{name} expects {n} arguments, got {len(args)}")

        if name in {"add", "sub", "mul"}:
            require(2)
            symbol = {"add": "+", "sub": "-", "mul": "*"}[name]
            return f"(({args[0]}) {symbol} ({args[1]}))"

        if name == "div":
            require(2)
            return f"(({args[0]}) / NULLIF(({args[1]}), 0))"

        if name == "neg":
            require(1)
            return f"(-({args[0]}))"

        if name == "abs":
            require(1)
            return f"ABS({args[0]})"

        if name == "log":
            require(1)
            # FactorMiner log is the natural logarithm.  DAI's log() defaults to
            # base 10, so use ln() explicitly and guard the domain.
            return (
                "LN(CASE WHEN "
                f"({args[0]}) > 0 THEN ({args[0]}) ELSE NULL END)"
            )

        if name == "sqrt":
            require(1)
            return (
                "SQRT(CASE WHEN "
                f"({args[0]}) >= 0 THEN ({args[0]}) ELSE NULL END)"
            )

        if name == "square":
            require(1)
            return f"POW(({args[0]}), 2)"

        if name == "power":
            require(2)
            return f"POW(({args[0]}), ({args[1]}))"

        if name == "signed_power":
            require(2)
            return f"SIGNEDPOWER(({args[0]}), ({args[1]}))"

        if name in {"tanh", "exp"}:
            require(1)
            return f"{name.upper()}({args[0]})"

        if name == "delay":
            require(2)
            return f"M_LAG(({args[0]}), {args[1]})"

        if name == "delta":
            require(2)
            return f"M_DELTA(({args[0]}), {args[1]})"

        if name == "ts_mean":
            require(2)
            return f"M_AVG(({args[0]}), {args[1]})"

        if name == "ts_sum":
            require(2)
            return f"M_SUM(({args[0]}), {args[1]})"

        if name == "ts_std":
            require(2)
            return f"M_STDDEV(({args[0]}), {args[1]})"

        if name == "ts_rank":
            require(2)
            return f"M_PCT_RANK(({args[0]}), {args[1]})"

        if name == "ts_decay":
            require(2)
            return f"M_DECAY_LINEAR(({args[0]}), {args[1]})"

        if name == "ema":
            require(2)
            return f"M_TA_EMA(({args[0]}), {args[1]})"

        if name == "wma":
            require(2)
            return f"M_TA_WMA(({args[0]}), {args[1]})"

        if name == "corr":
            require(3)
            return f"M_CORR(({args[0]}), ({args[1]}), {args[2]})"

        if name == "slope":
            require(2)
            time_index = "DATE_DIFF('day', DATE '1970-01-01', date)"
            return f"M_REGR_SLOPE(({args[0]}), {time_index}, {args[1]})"

        if name == "cs_rank":
            require(1)
            return f"C_PCT_RANK(({args[0]}))"

        if name == "cs_zscore":
            require(1)
            return f"C_ZSCORE(({args[0]}))"

        if name == "scale":
            require(1)
            return f"C_SCALE(({args[0]}), 1.0)"

        if name == "greater":
            require(2)
            return f"(({args[0]}) > ({args[1]}))"

        if name == "if_else":
            require(3)
            return (
                f"(CASE WHEN ({args[0]}) THEN ({args[1]}) "
                f"ELSE ({args[2]}) END)"
            )

        raise ValueError(f"Unsupported operator: {name}")

    tree = ast.parse(expression, mode="eval")
    return compile_node(tree.body)


def main(datasources, start_date, end_date):
    """Build one daily factor and return exactly date/instrument/factor."""

    import numpy as np
    import pandas as pd
    import dai

    if "bar1m" not in datasources:
        raise KeyError("datasources must contain the replaceable 'bar1m' table")

    bar1m = datasources["bar1m"]
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_.]*", bar1m):
        raise ValueError("Unexpected bar1m table name")

    factor_sql = _expression_to_sql(FACTOR_EXPRESSION)
    direction = 1.0 if TRAIN_IC >= 0 else -1.0

    eval_start = pd.to_datetime(start_date).normalize()
    eval_end = pd.to_datetime(end_date).normalize()
    query_start = eval_start - pd.Timedelta(days=BUFFER_DAYS)

    # The minute-table volume field is cumulative within the trading day, so
    # MAX(volume), rather than SUM(volume), gives daily volume.
    sql = f"""
    WITH daily AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument::STRING AS instrument,
            FIRST(open ORDER BY date) AS open,
            MAX(high) AS high,
            MIN(low) AS low,
            LAST(close ORDER BY date) AS close,
            FIRST(pre_close ORDER BY date) AS pre_close,
            MAX(volume) AS volume
        FROM {bar1m}
        WHERE open > 0 AND high > 0 AND low > 0 AND close > 0
        GROUP BY date::DATE, instrument
    ), features AS (
        SELECT
            date,
            instrument,
            open,
            high,
            low,
            close,
            volume,
            close / NULLIF(pre_close, 0) - 1.0 AS daily_return
        FROM daily
    )
    SELECT
        date,
        instrument,
        {direction} * CAST(({factor_sql}) AS DOUBLE) AS factor
    FROM features
    ORDER BY instrument, date
    """

    factor_data = dai.query(
        sql,
        filters={
            "date": [
                query_start.strftime("%Y-%m-%d %H:%M:%S"),
                pd.to_datetime(end_date).strftime("%Y-%m-%d %H:%M:%S"),
            ]
        },
        compression=True,
    ).df()

    factor_data["date"] = pd.to_datetime(factor_data["date"]).dt.normalize()
    factor_data["instrument"] = factor_data["instrument"].astype("string")
    factor_data["factor"] = pd.to_numeric(
        factor_data["factor"], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)
    factor_data = factor_data[
        (factor_data["date"] >= eval_start)
        & (factor_data["date"] <= eval_end)
    ]

    # Restrict the output to the official historical CSI 1000 constituent pool.
    stock_pool = dai.query(
        """
        SELECT
            date::DATE::DATETIME AS date,
            instrument::STRING AS instrument
        FROM bigalpha_2026_instruments
        ORDER BY date, instrument
        """,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    stock_pool["date"] = pd.to_datetime(stock_pool["date"]).dt.normalize()
    stock_pool["instrument"] = stock_pool["instrument"].astype("string")
    stock_pool = stock_pool.drop_duplicates(["date", "instrument"])

    result = stock_pool.merge(
        factor_data[["date", "instrument", "factor"]],
        how="inner",
        on=["date", "instrument"],
        validate="one_to_one",
    )
    result = result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"], ignore_index=True
    )

    if list(result.columns) != ["date", "instrument", "factor"]:
        raise RuntimeError("Invalid submission columns")
    if result.empty:
        raise RuntimeError("Factor output is empty")
    return result

